### Convert Keras Models to ONNX

The below cell converts every `.keras` model in the `models/` directory to ONNX format, saving each `.ONNX` file in the same folder with the original filename. It automatically derives the input signature from each model, uses ONNX opset 13, and validates each converted model. Conversion errors are reported individually so the remaining models can still be processed. Original Keras files are preserved.

In [1]:
from pathlib import Path

import tensorflow as tf
import onnx
import tf2onnx


MODELS_DIR = Path("models")
OPSET = 17 # Matches the torch script, if changed here, please change there too

keras_files = sorted(MODELS_DIR.glob("*.keras"))

if not keras_files:
    print(f"No .keras models found in {MODELS_DIR.resolve()}")

for model_path in keras_files:
    new_name = model_path.name.replace(
        "ai_capstone_keras_",
        "Tensorflow_keras_",
        1
    )

    onnx_path = model_path.with_name(new_name).with_suffix(".onnx")

    print(f"\nConverting: {model_path.name}")

    try:
        model = tf.keras.models.load_model(
            model_path,
            compile=False,
        )

        # Build an input signature from the model's actual input shape.
        input_signature = [
            tf.TensorSpec(
                shape=tuple(
                    None if dim is None else int(dim)
                    for dim in tensor.shape
                ),
                dtype=tensor.dtype,
                name=f"input_{i}",
            )
            for i, tensor in enumerate(model.inputs)
        ]

        @tf.function(input_signature=input_signature)
        def serving_fn(*inputs):
            result = model(
                inputs[0] if len(inputs) == 1 else list(inputs),
                training=False,
            )
            return result

        onnx_model, _ = tf2onnx.convert.from_function(
            serving_fn,
            input_signature=input_signature,
            opset=OPSET,
            output_path=str(onnx_path),
        )

        # Validate the generated ONNX model.
        onnx.checker.check_model(onnx_model)

        print(f"SUCCESS: {onnx_path.name}")

    except Exception as exc:
        print(f"FAILED: {model_path.name}")
        print(f"{type(exc).__name__}: {exc}")

print("\nConversion run complete.")


2026-10-22 11:39:28.173547: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-22 11:39:28.190291: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-22 11:39:28.195365: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-22 11:39:28.207585: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.



Converting: ai_capstone_keras_CNN_medium.keras


I0000 00:00:1792669169.697472    6722 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1792669169.706620    6722 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1792669169.709130    6722 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1792669169.712360    6722 cuda_executor.cc:1015] successful NUMA node read from SysFS ha

SUCCESS: Tensorflow_keras_CNN_medium.onnx

Converting: ai_capstone_keras_CNN_small.keras


I0000 00:00:1792669170.768377    6722 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1792669170.769314    6722 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-10-22 11:39:30.769383: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
I0000 00:00:1792669170.769583    6722 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1792669170.770457    6722 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so

SUCCESS: Tensorflow_keras_CNN_small.onnx

Converting: ai_capstone_keras_hybrid.keras


/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:372: UserWarning: `build()` was called on layer 'transformer_block', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
I0000 00:00:1792669171.205092    6722 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
I0000 00:00:1792669171.206029    6722 devices.cc:67] Number of eligible GPUs (core count >= 8, compute capability >= 0.0): 1
2026-10-22 11:39:31.206114: I tensorflow/core/grappler/clusters/single_machine.cc:361] Starting new session
I0000 00:00:1792669171.206333    6722 c

SUCCESS: Tensorflow_keras_hybrid.onnx

Converting: ai_capstone_keras_pure_vit_model.keras


/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:372: UserWarning: `build()` was called on layer 'patch_embedding', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:372: UserWarning: `build()` was called on layer 'transformer_block_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
I0000 00:00:1792669171.912813    6722 cuda_executor.cc:1015] successful NUMA node read from SysFS had negative value (-1), but there must be at 

SUCCESS: Tensorflow_keras_pure_vit_model.onnx

Conversion run complete.
